In [ ]:
import json

OBJECT_ID = "4001732069"

with open("cleaned_apartements_processed.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]
    processed = {str(r.get("object_id")): r for r in records}

record = processed.get(OBJECT_ID)
if record is None:
    print("Not found.")
else:
    # Build a mapping from image path to bin label
    bin_map = record.get("clip_bin_map", {})
    path_to_label = {img["path"]: label for label, imgs in bin_map.items() for img in imgs}

    for img in record.get("clip_selected_images", []):
        label = path_to_label.get(img["path"], "?")
        print(label, img["confidence"], img["path"])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

if record is not None:
    bin_map = record.get("clip_bin_map", {})
    path_to_label = {img["path"]: label for label, imgs in bin_map.items() for img in imgs}

    # Collect up to 6 existing images and their info
    images_to_plot = []
    for img in record.get("clip_selected_images", []):
        path = img["path"]
        if path.startswith("preprocess_pipeline/"):
            path = path[len("preprocess_pipeline/"):]
        label = path_to_label.get(img["path"], "?")
        conf = img["confidence"]
        if Path(path).is_file():
            images_to_plot.append((path, label, conf))
        if len(images_to_plot) == 6:
            break

    if images_to_plot:
        fig, axes = plt.subplots(2, 3, figsize=(12, 8))
        axes = axes.flatten()
        for ax, (path, label, conf) in zip(axes, images_to_plot):
            image = Image.open(path)
            ax.imshow(image)
            ax.set_title(f"{label} ({conf:.2f})")
            ax.axis("off")
        # Hide unused axes
        for ax in axes[len(images_to_plot):]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("No images to plot.")